# ⚠️ Manejo de Excepciones aplicado al CRUD
## Algoritmos y Estructuras de Datos I — UADE

---

En este notebook vamos a aprender a manejar errores en Python aplicándolo directamente a la estructura del **Proyecto Final Integrador**.

El proyecto está organizado en tres módulos:

| Módulo | Responsabilidad |
|---|---|
| `utilities.py` | Validaciones con regex (`validar_nombre`, `validar_precio_unitario`, etc.) |
| `inputHandler.py` | Lectura y normalización del input del usuario |
| `services.py` | Lógica de negocio: persistir, buscar, generar IDs |
| `main.py` | Menú principal, manejo de excepciones de alto nivel |

> 💡 Las excepciones permiten que el programa **no se caiga** ante un error, sino que lo maneje de forma controlada y le informe al usuario qué ocurrió.


---
## Bloque 1 — Tipos de error en Python

Antes de manejar excepciones, es importante reconocer los distintos tipos de error:

| Tipo | Cuándo ocurre | Ejemplo en el proyecto |
|---|---|---|
| **SyntaxError** | El código está mal escrito | Falta un `:` en un `for` |
| **ValueError** | El valor no es del tipo esperado | `int("abc")` al convertir el ID |
| **TypeError** | Operación sobre tipo incorrecto | `float(None)` en el precio |
| **KeyError** | Clave inexistente en un dict | `producto["precio"]` si la clave es `"precio_unitario"` |
| **IndexError** | Índice fuera de rango en una lista | Acceder a `productos[10]` si hay solo 2 |

Los **SyntaxError** se detectan antes de ejecutar. Los demás ocurren en tiempo de ejecución y son los que necesitamos manejar con `try/except`.


### 1.1 — ValueError: el más común en ingreso de datos

Ocurre cuando intentamos convertir un string a un tipo numérico y el valor no es válido.  
En `main.py` esto aparece al leer el `id_producto` para la búsqueda:

```python
id_producto = int(input("Ingrese el id_producto a buscar: "))
```

Si el usuario escribe `"abc"`, Python lanza un `ValueError`.


In [ ]:
# Simulamos el error sin try/except
entrada = "abc"
id_producto = int(entrada)   # ← ValueError: invalid literal for int()

### 1.2 — KeyError: acceso a clave inexistente en un diccionario


In [ ]:
# Simulamos el error
producto = {"id_producto": 1, "nombre": "Lápiz negro HB", "precio_unitario": 350}

# Accedemos a una clave que no existe
print(producto["precio"])   # ← KeyError: 'precio'

---
## Bloque 2 — try / except

La estructura básica para manejar excepciones:

```python
try:
    # código que puede fallar
except TipoDeError as e:
    # qué hacer si falla
finally:
    # se ejecuta siempre, haya error o no (opcional)
```

- El bloque `try` se ejecuta normalmente.
- Si ocurre un error, Python salta al `except` correspondiente.
- El `finally` se ejecuta siempre, con o sin error (útil para cerrar archivos, conexiones, etc.).


### 2.1 — Manejo del ValueError en main.py

Este es exactamente el patrón que usa el proyecto en la opción 3 del menú (buscar por ID):


In [ ]:
# Patrón usado en main.py — case "3"
entrada = input("Ingrese el id_producto a buscar: ")

try:
    id_producto = int(entrada)
    print(f"ID ingresado correctamente: {id_producto}")
except ValueError:
    print("ID inválido, debe ser un número entero.")

### 2.2 — Capturar múltiples excepciones

Dentro de un mismo bloque `try` pueden ocurrir errores de distinto tipo.  
Usamos varios `except` para manejar cada caso por separado:


In [ ]:
productos = [
    {"id_producto": 1, "categoria": "Escritura", "nombre": "Lápiz negro HB", "precio_unitario": 350},
    {"id_producto": 2, "categoria": "Escritura", "nombre": "Bolígrafo azul",  "precio_unitario": 500},
]

entrada = input("Ingrese el índice del producto: ")

try:
    indice = int(entrada)
    producto = productos[indice]
    print(f"Producto: {producto['nombre']} — ${producto['precio_unitario']}")
except ValueError:
    print("Error: el índice debe ser un número entero.")
except IndexError:
    print(f"Error: índice fuera de rango. Hay {len(productos)} productos (índices 0 a {len(productos)-1}).")
except KeyError as e:
    print(f"Error: el producto no tiene la clave {e}.")

### 2.3 — Capturar cualquier excepción con `Exception`

En `main.py` el bloque `try` del case `"1"` captura cualquier excepción con `Exception as e`.  
Esto es útil cuando queremos que **ningún error rompa el programa**, sea cual sea el tipo.

```python
# En main.py — case "1"
try:
    producto = inputHandler.leer_producto()
    response = services.persistir_producto(productos, producto)
    print("Producto insertado exitosamente") if response else None
except Exception as e:
    print(f"Error al crear el producto: {e}")
```

> ⚠️ Capturar `Exception` genérico está bien en el nivel más alto del programa (main).  
> En funciones internas es mejor capturar el tipo específico para no ocultar errores inesperados.


---
## Bloque 3 — Lanzar excepciones con `raise`

`raise` se usa para **generar una excepción intencionalmente** cuando una regla de negocio no se cumple.  
Es la forma de comunicar errores desde las funciones internas hacia el `try/except` del nivel superior.

```python
raise ValueError("El precio debe ser mayor a cero.")
raise TypeError("El parámetro debe ser una lista.")
raise Exception("Mensaje de error genérico.")
```


### 3.1 — raise en inputHandler.py

En `inputHandler.py`, cuando el usuario supera el máximo de intentos para ingresar un dato válido, se lanza un `ValueError` con `raise`:

```python
for intento in range(MAX_INTENTOS):
    precio_unitario = input("Ingrese el precio unitario: ")
    if utilities.validar_precio_unitario(precio_unitario):
        break
    print("Precio no válido, intente nuevamente...")
else:
    raise ValueError("Se superó el máximo de intentos para precio unitario.")
```

> 💡 El `else` de un `for` se ejecuta **solo si el bucle terminó sin un `break`**.  
> Es decir: si el usuario nunca ingresó un valor válido en los 3 intentos.

Ese `ValueError` sube hasta el `try/except` de `main.py` que lo captura con `except Exception as e`.


In [ ]:
import re

# Simulamos utilities.validar_precio_unitario
def validar_precio_unitario(precio):
    patron = r"[0-9]+(\.[0-9]{1,2})?"
    return bool(re.fullmatch(patron, precio.strip()))

# Simulamos inputHandler.leer_precio con raise
def leer_precio():
    MAX_INTENTOS = 3
    for intento in range(MAX_INTENTOS):
        precio = input(f"Intento {intento+1}/{MAX_INTENTOS} — Ingrese el precio unitario: ")
        if validar_precio_unitario(precio):
            return float(precio.strip())
        print("Precio no válido, intente nuevamente...")
    else:
        raise ValueError("Se superó el máximo de intentos para precio unitario.")

# Simulamos el try/except de main.py
try:
    precio = leer_precio()
    print(f"Precio ingresado: ${precio:.2f}")
except Exception as e:
    print(f"Error al ingresar el producto: {e}")

### 3.2 — raise en services.py

Podemos agregar validaciones con `raise` en `services.py` para proteger la integridad de los datos antes de persistirlos.  
Por ejemplo, verificar que el producto no tenga nombre duplicado:


In [ ]:
import re

productos = [
    {"id_producto": 1, "categoria": "Escritura", "nombre": "Lápiz negro HB", "precio_unitario": 350},
    {"id_producto": 2, "categoria": "Escritura", "nombre": "Bolígrafo azul",  "precio_unitario": 500},
]

def generar_id_producto(productos):
    if not productos:
        return 1
    return max(productos, key=lambda p: p["id_producto"])["id_producto"] + 1

def persistir_producto(productos, producto):
    # Validación de negocio: nombre duplicado
    nombres_existentes = [p["nombre"].lower() for p in productos]
    if producto["nombre"].lower() in nombres_existentes:
        raise ValueError(f"Ya existe un producto con el nombre '{producto['nombre']}'.")

    producto["id_producto"] = generar_id_producto(productos)
    productos.append(producto)
    return True

# Probamos con nombre duplicado
try:
    nuevo = {"categoria": "Escritura", "nombre": "Lápiz negro HB", "precio_unitario": 400}
    persistir_producto(productos, nuevo)
    print("Producto insertado.")
except ValueError as e:
    print(f"Error al crear el producto: {e}")

# Probamos con nombre nuevo
try:
    nuevo = {"categoria": "Escritura", "nombre": "Marcador permanente", "precio_unitario": 1200}
    persistir_producto(productos, nuevo)
    print(f"Producto insertado. Total: {len(productos)} productos.")
except ValueError as e:
    print(f"Error al crear el producto: {e}")

---
## Bloque 4 — `while True`, `break` y `continue` en el menú

En `main.py` el menú usa `while True` para mantenerse activo hasta que el usuario elija salir.  
`break` y `continue` controlan el flujo dentro del loop:

| Sentencia | Efecto |
|---|---|
| `break` | Termina el loop inmediatamente (case `"5"` — salir) |
| `continue` | Salta al inicio del loop, ignorando el resto del bloque actual |

En el case `"3"` de `main.py`, si el ID ingresado no es válido se usa `continue` para volver al menú sin ejecutar la búsqueda:

```python
case "3":
    try:
        id_producto = int(input("Ingrese el id_producto a buscar: "))
    except ValueError:
        print("ID inválido, debe ser un número entero.")
        continue    # ← vuelve al inicio del while, muestra el menú de nuevo
    ...
```


In [ ]:
# Simulamos el flujo del menú con while True, break y continue

productos = [
    {"id_producto": 1, "nombre": "Lápiz negro HB",  "precio_unitario": 350},
    {"id_producto": 2, "nombre": "Bolígrafo azul",   "precio_unitario": 500},
]

def get_producto_by_id(productos, id_producto):
    return [p for p in productos if p["id_producto"] == id_producto]

while True:
    print("\n1. Buscar por ID\n2. Salir\n")
    opcion = input("Opción: ")

    match opcion:
        case "1":
            try:
                id_producto = int(input("Ingrese el id_producto: "))
            except ValueError:
                print("ID inválido, debe ser un número entero.")
                continue    # ← vuelve al menú

            resultado = get_producto_by_id(productos, id_producto)
            if resultado:
                print(f"Encontrado: {resultado[0]['nombre']}")
            else:
                print("No se encontró el producto.")

        case "2":
            print("Saliendo...")
            break           # ← termina el while

        case _:
            print("Opción inválida.")

---
## Bloque 5 — Flujo completo de excepciones en el proyecto

Veamos cómo fluyen las excepciones a través de los tres módulos:

```
main.py
  └── try/except (Exception)          ← captura cualquier error de nivel alto
        └── inputHandler.leer_producto()
              ├── for/else + raise ValueError  ← si se superan los intentos
              └── utilities.validar_*(...)     ← retorna True/False (sin excepciones)

main.py
  └── try/except (ValueError)         ← captura error de conversión de ID
        └── int(input(...))
```

La clave del diseño es la **separación de responsabilidades**:
- `utilities` valida y retorna `True`/`False` — **no lanza excepciones**
- `inputHandler` lanza `ValueError` si se agotan los intentos
- `services` lanza `ValueError` si se viola una regla de negocio
- `main` captura todo con `try/except` y decide qué mostrar al usuario


In [ ]:
import re

# ── utilities ────────────────────────────────────────────────────
def validar_categoria(categoria):
    return bool(re.fullmatch(r"[A-Za-záéíóúÁÉÍÓÚüÜñÑ\s]+", categoria.strip()))

def validar_nombre(nombre):
    return bool(re.fullmatch(r"[A-Za-z0-9áéíóúÁÉÍÓÚüÜñÑ ]+", nombre.strip()))

def validar_precio_unitario(precio):
    return bool(re.fullmatch(r"[0-9]+(\.[0-9]{1,2})?", precio.strip()))

# ── inputHandler ─────────────────────────────────────────────────
def leer_producto():
    MAX_INTENTOS = 3

    for _ in range(MAX_INTENTOS):
        categoria = input("Ingrese la categoría: ")
        if validar_categoria(categoria): break
        print("Categoría no válida, intente nuevamente...")
    else:
        raise ValueError("Se superó el máximo de intentos para categoría.")

    for _ in range(MAX_INTENTOS):
        nombre = input("Ingrese el nombre: ")
        if validar_nombre(nombre): break
        print("Nombre no válido, intente nuevamente...")
    else:
        raise ValueError("Se superó el máximo de intentos para nombre.")

    for _ in range(MAX_INTENTOS):
        precio = input("Ingrese el precio unitario: ")
        if validar_precio_unitario(precio): break
        print("Precio no válido, intente nuevamente...")
    else:
        raise ValueError("Se superó el máximo de intentos para precio unitario.")

    return {
        "categoria":       categoria.strip().title(),
        "nombre":          nombre.strip().title(),
        "precio_unitario": float(precio.strip()),
    }

# ── services ─────────────────────────────────────────────────────
def generar_id(productos):
    return max((p["id_producto"] for p in productos), default=0) + 1

def persistir_producto(productos, producto):
    nombres = [p["nombre"].lower() for p in productos]
    if producto["nombre"].lower() in nombres:
        raise ValueError(f"Ya existe un producto con el nombre '{producto['nombre']}'.")
    producto["id_producto"] = generar_id(productos)
    productos.append(producto)
    return True

# ── main ─────────────────────────────────────────────────────────
productos = [
    {"id_producto": 1, "categoria": "Escritura", "nombre": "Lápiz negro HB", "precio_unitario": 350},
    {"id_producto": 2, "categoria": "Escritura", "nombre": "Bolígrafo azul",  "precio_unitario": 500},
]

print("=== Simulación del case '1' — Alta de producto ===")
try:
    producto = leer_producto()
    response = persistir_producto(productos, producto)
    print("Producto insertado exitosamente.") if response else None
except Exception as e:
    print(f"Error al crear el producto: {e}")

print(f"Productos en lista: {len(productos)}")
for p in productos:
    print(f"  [{p['id_producto']}] {p['nombre']} — ${p['precio_unitario']:.2f}")

---
## 🏋️ Ejercicios prácticos


### Ejercicio 1 — Agregar validación con `raise` en `services.py`

Extendé la función `persistir_producto()` para que también valide que el `precio_unitario` sea mayor a cero.  
Si no lo es, debe lanzar un `ValueError` con un mensaje descriptivo.

> El `try/except` en `main.py` ya captura ese error, no hace falta modificarlo.


In [ ]:
import re

def validar_precio_unitario(precio):
    return bool(re.fullmatch(r"[0-9]+(\.[0-9]{1,2})?", str(precio).strip()))

def persistir_producto(productos, producto):
    # Completá: lanzá ValueError si precio_unitario <= 0
    # __________

    nombres = [p["nombre"].lower() for p in productos]
    if producto["nombre"].lower() in nombres:
        raise ValueError(f"Ya existe un producto con el nombre '{producto['nombre']}'.")

    producto["id_producto"] = max((p["id_producto"] for p in productos), default=0) + 1
    productos.append(producto)
    return True

# Casos de prueba — no modificar
productos = []

try:
    persistir_producto(productos, {"nombre": "Lápiz", "categoria": "Escritura", "precio_unitario": 0})
except ValueError as e:
    print(f"✅ Capturado correctamente: {e}")

try:
    persistir_producto(productos, {"nombre": "Lápiz", "categoria": "Escritura", "precio_unitario": 350})
    print(f"✅ Insertado correctamente. Total: {len(productos)}")
except ValueError as e:
    print(f"❌ No debería fallar: {e}")

### Ejercicio 2 — Manejo de excepciones en búsqueda por ID

En `main.py` la búsqueda por ID ya maneja el `ValueError` de conversión.  
Extendé el bloque para que también maneje el caso en que `get_producto_by_id()` lanza un `TypeError` si el ID es negativo.

Completá la función `get_producto_by_id()` para que lance `TypeError` si `id_producto <= 0`, y agregá el `except TypeError` correspondiente en el bloque de llamada.


In [ ]:
def get_producto_by_id(productos, id_producto):
    # Completá: lanzá TypeError si id_producto <= 0
    # __________
    return [p for p in productos if p["id_producto"] == id_producto]

productos = [
    {"id_producto": 1, "nombre": "Lápiz negro HB",  "precio_unitario": 350},
    {"id_producto": 2, "nombre": "Bolígrafo azul",   "precio_unitario": 500},
]

# Bloque de llamada — completá el except que falta
entrada = input("Ingrese el id_producto a buscar: ")
try:
    id_producto = int(entrada)
    resultado = get_producto_by_id(productos, id_producto)
    print(f"Resultado: {resultado if resultado else 'No encontrado'}")
except ValueError:
    print("ID inválido, debe ser un número entero.")
# ← agregá el except para TypeError acá

### 🏆 Ejercicio 3 — Desafío: `leer_producto()` con registro de intentos fallidos

Modificá `leer_producto()` para que, además de lanzar `ValueError` al superar los intentos, **imprima un resumen** al final indicando cuántos intentos fallidos hubo por cada campo.

Ejemplo de salida esperada si el usuario falla 2 veces en precio:
```
Resumen de intentos fallidos:
  - categoria:       0 intento/s fallido/s
  - nombre:          0 intento/s fallido/s
  - precio_unitario: 2 intento/s fallido/s
```


In [ ]:
import re

def validar_categoria(c):    return bool(re.fullmatch(r"[A-Za-záéíóúÁÉÍÓÚüÜñÑ\s]+", c.strip()))
def validar_nombre(n):       return bool(re.fullmatch(r"[A-Za-z0-9áéíóúÁÉÍÓÚüÜñÑ ]+", n.strip()))
def validar_precio(p):       return bool(re.fullmatch(r"[0-9]+(\.[0-9]{1,2})?", p.strip()))

def leer_producto():
    MAX_INTENTOS = 3
    intentos_fallidos = {"categoria": 0, "nombre": 0, "precio_unitario": 0}

    # Tu solución acá 👇
    # Modificá los tres bloques for/else para registrar los intentos fallidos
    # y mostrar el resumen al final (antes del return o del raise)


# Llamada de prueba
try:
    producto = leer_producto()
    print(f"Producto cargado: {producto}")
except ValueError as e:
    print(f"Error: {e}")

---
## 💬 Reflexión

Respondé en esta celda (doble clic para editar):

**1. ¿Por qué `utilities.py` retorna `True`/`False` en lugar de lanzar excepciones?**

> _Tu respuesta acá_

**2. ¿Cuál es la diferencia entre capturar `ValueError` específico y capturar `Exception` genérico? ¿Cuándo usarías cada uno?**

> _Tu respuesta acá_

**3. ¿Qué pasaría si en `main.py` no hubiera ningún `try/except` y el usuario ingresa un ID con letras?**

> _Tu respuesta acá_
